In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lpips

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time

In [ ]:
train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [ ]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [ ]:
class DTD_Dataset(Dataset):

    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

In [ ]:
def nonlinearity(x):
  return x * torch.sigmoid(x)


def Normalize(in_channels):
  return torch.nn.GroupNorm(num_groups=32, num_channels=in_channels, eps=1e-6, affine=True)

class Upsample(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
          self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)

    def forward(self,x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if self.with_conv:
          x = self.conv(x)
        return x

class Downsample(nn.Module):
    def __init__(self, in_channels, with_conv=True):
        super().__init__()
        self.with_conv = with_conv
        if with_conv:
          self.conv = torch.nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)

    def forward(self, x):
        if self.with_conv:
          x = F.pad(x, (0, 1, 0, 1), mode="constant", value=0)
          x = self.conv(x)
        else:
          x = F.avg_pool2d(x, kernel_size=2, stride=2)
        return x

class ResnetBlock(nn.Module):
    def __init__(self, in_channels, out_channels=None, dropout=0.0):
        super().__init__()
        out_channels = in_channels if out_channels is None else out_channels

        self.norm1 = Normalize(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)

        self.norm2 = Normalize(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)

        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.shortcut = nn.Identity()



    def forward(self,x):
        h = self.norm1(x)
        h = nonlinearity(h)
        h = self.conv1(h)

        h = self.norm2(h)
        h = nonlinearity(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return self.shortcut(x) + h

class AttnBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.norm = Normalize(channels)

        self.q = nn.Conv2d(channels, channels, kernel_size=1)

        self.k = nn.Conv2d(channels, channels, kernel_size=1)

        self.v = nn.Conv2d(channels, channels, kernel_size=1)

        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)


    def forward(self,x):
        h_ = self.norm(x)

        q = self.q(h_)
        k = self.k(h_)
        v = self.v(h_)


        b,c,h,w = q.shape

        q = q.reshape(b,c,h*w)
        q = q.permute(0,2,1)

        k = k.reshape(b,c,h*w)

        attention = torch.bmm(q,k)
        attention = attention * (c**-0.5)
        attention = F.softmax(attention, dim=2)

        v = v.reshape(b,c,h*w)

        attention = attention.permute(0,2,1)

        out = torch.bmm(v,attention)
        out = out.reshape(b,c,h,w)

        out = self.proj_out(out)

        return x + out

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), resolution=256, z_channels=256, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.ch = ch
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution

        self.conv_in = nn.Conv2d(in_channels=in_channels, out_channels=ch, kernel_size=3, stride=1, padding=1)
        curr_resolution = resolution

        self.down = nn.ModuleList()
        in_ch_mult = (1,) + tuple(ch_mult)

        for i_level in range(self.num_resolutions):
          block = nn.ModuleList()
          attn = nn.ModuleList()

          block_in = ch * in_ch_mult[i_level]
          block_out = ch * ch_mult[i_level]

          for i_block in range(self.num_res_blocks):
            block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, dropout=dropout))

            block_in = block_out

            if curr_resolution in attn_resolutions:
              attn.append(AttnBlock(block_in))

          down = nn.Module()
          down.block = block
          down.attn = attn

          # do not downsample at the final resolution
          if i_level != self.num_resolutions - 1:
            down.downsample = Downsample(block_in, resamp_with_conv)
            curr_resolution = curr_resolution // 2

          self.down.append(down)

        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)
        self.mid.attn_1 = AttnBlock(block_in)

        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)

        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, z_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
      h = self.conv_in(x)                             # 256x256x3
      for i_level in range(self.num_resolutions):
        for i_block in range(self.num_res_blocks):
          h = self.down[i_level].block[i_block](h)  # 256x256x128 -> 128x128x128 -> 64x64x128 -> 32x32x256 -> 16x16x512
          if len(self.down[i_level].attn) > 0:
            h = self.down[i_level].attn[i_block](h)

        if i_level != self.num_resolutions - 1:
            h = self.down[i_level].downsample(h)

      h = self.mid.block_1(h)
      h = self.mid.attn_1(h)
      h = self.mid.block_2(h)

      h = self.norm_out(h)
      h = nonlinearity(h)
      h = self.conv_out(h)                            # 16x16x256
      return h

In [ ]:
class Decoder(nn.Module):
    def __init__(self, out_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), resolution=256, z_channels=256, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.ch = ch
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution

        block_in = ch * ch_mult[-1]
        curr_resolution = resolution // (2 ** (self.num_resolutions -1))

        self.conv_in = nn.Conv2d(z_channels, block_in, kernel_size=3, stride=1, padding=1)
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)
        self.mid.attn_1 = AttnBlock(block_in)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, dropout=dropout)

        self.up = nn.ModuleList()

        for i_level in reversed(range(self.num_resolutions)):
          block = nn.ModuleList()
          attn = nn.ModuleList()

          block_out = ch * ch_mult[i_level]

          for i_block in range(num_res_blocks + 1):
            block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, dropout=dropout))

            block_in = block_out

            if curr_resolution in attn_resolutions:
              attn.append(AttnBlock(block_in))

          up = nn.Module()
          up.block = block
          up.attn = attn

          # upsample except at final stage
          if i_level != 0:
            up.upsample = Upsample(block_in, resamp_with_conv)
            curr_resolution = curr_resolution * 2

          self.up.append(up)

        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
      h = self.conv_in(x)

      h = self.mid.block_1(h)
      h = self.mid.attn_1(h)
      h = self.mid.block_2(h)

      for i_level in range(self.num_resolutions):
        for i_block in range(self.num_res_blocks + 1):
          h = self.up[i_level].block[i_block](h)
          if len(self.up[i_level].attn) > 0:
            h = self.up[i_level].attn[i_block](h)
        if i_level != self.num_resolutions - 1:
          h = self.up[i_level].upsample(h)

      h = self.norm_out(h)
      h = nonlinearity(h)
      h = self.conv_out(h)
      return h

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_permuted = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_permuted.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        active_codes = torch.unique(encoding_indices).numel()

        return quantized, loss, perplexity, active_codes


In [ ]:
class VQGAN(nn.Module):
    def __init__(self, resolution=256, in_channels=3, out_channels=3, ch=128, ch_mult=(1, 1, 2, 2, 4), num_res_blocks=2, attn_resolutions=(16,), z_channels=256, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25, dropout=0.0, resamp_with_conv=True):
        super().__init__()
        self.encoder = Encoder(in_channels, ch, ch_mult, num_res_blocks, attn_resolutions, resolution, z_channels, dropout, resamp_with_conv)
        self.quant_conv = nn.Conv2d(z_channels, embedding_dim,kernel_size=1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, z_channels,kernel_size=1)
        self.decoder = Decoder(out_channels, ch, ch_mult, num_res_blocks, attn_resolutions, resolution, z_channels, dropout, resamp_with_conv)

    def forward(self, x):
        z = self.encoder(x)
        z = self.quant_conv(z)

        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        quantized = self.post_quant_conv(quantized)

        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity, active_codes

VQGAN:
- Generator: Encoder-VQ-Decoder. It takes a real texture image and tries to generate a flawless, high-resolution copy of it
- Discriminator: it looks at the original image and the generated copy, trying to figure out which one is the fake one

Perceptual Loss (Reconstruction Loss): LPIPS (Learned Perceptual Image Patch Similarity). It does not look at the raw pixel coords, but the real texture and the generated texture through a pre-trained net that knows how to recognize shapes, edges and textures. It compares the hidden feature layers (it looks at the same freq of lines, same style of roughness, etc.)

VGG16: we freeze weights because when Perceptual Loss is high and the gradient flows backwards, only the Generator is updates. It's a pretrained net, we are only using its specialized vision

$Loss_{VQGAN} = L_{recon} + \lambda \cdot L_{GAN} + L_{percep} + L_{codebook}$

where

- $L_{GAN}$ uses the Hinge Loss for both generator and discriminator, with adaptive weighting: $\lambda = \frac{\nabla _{G_{L}} [L_{recon} + L_{percep}]}{\nabla _{G_{L}} [L_{GAN} + ϵ]}$


In [ ]:
class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1)              # 512x512 -> 256x256
        self.conv2 = nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1)     # 256x256 -> 128x128
        self.conv3 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 64x64
        self.downsample = nn.Conv2d(hidden_dim * 2, embedding_dim, kernel_size=4, stride=4, padding=0)  # 64x64 -> 16x16

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.downsample(x)
        return x

class Decoder(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(embedding_dim, hidden_dim * 2, kernel_size=4, stride=4, padding=0) #16x16 -> 64x64
        self.conv1 = nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1)       # 64x64 -> 128x128
        self.conv2 = nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 256x256
        self.conv3 = nn.ConvTranspose2d(hidden_dim // 2, 3, kernel_size=4, stride=2, padding=1)               # 256x256 -> 512x512

    def forward(self, x):
        x = self.upsample(x)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        x = torch.tanh(x)
        return x

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        return quantized, loss


In [ ]:
class VQGAN(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256, num_embeddings=1024, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity, active_codes

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim, kernel_size=4, stride=2, padding=1)                   # (3, 256, 256) -> (64, 128, 128)
        self.relu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # (64, 128, 128) -> (128, 64, 64)
        self.norm1 = nn.BatchNorm2d(hidden_dim * 2)

        self.conv3 = nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)  # (128, 64, 64) -> (256, 32, 32)
        self.norm2 = nn.BatchNorm2d(hidden_dim * 4)

        self.conv4 = nn.Conv2d(hidden_dim * 4, 1, kernel_size=4, stride=1, padding=1)               # (256, 32, 32) -> (1, 31, 31)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.norm1(self.conv2(x)))
        x = self.relu(self.norm2(self.conv3(x)))
        x = self.conv4(x)
        return x

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, real, fake):
        return F.mse_loss(self.slice(fake), self.slice(real))

In [ ]:
def calculate_adaptive_weight(recon_loss, g_loss, last_layer_weights):
    recon_grads = torch.autograd.grad(recon_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]

    if recon_grads is None:
      recon_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      recon_norm = torch.norm(recon_grads)

    if g_grads is None:
      g_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      g_norm = torch.norm(g_grads)

    lambda_weight = recon_norm / (g_norm + 1e-4)
    lambda_weight = torch.clamp(lambda_weight, 0.0, 1e4).detach()
    return lambda_weight

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = VQGAN().to(device)
discriminator = Discriminator().to(device)
optimizer_generator = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=5e-5, betas=(0.5, 0.9))
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)
#perceptual_loss_fn = PerceptualLoss().to(device)

In [ ]:
def set_requires_grad(model, requires_grad):
  for param in model.parameters():
    param.requires_grad = requires_grad

def train_step(generator, discriminator, perceptual_loss_fn, data, optimizer_g, optimizer_d, disc_start_step, current_global_step):
    # GENERATOR

    # disable discriminator gradients while training generator
    set_requires_grad(discriminator, False)

    optimizer_g.zero_grad()
    recon_batch, vq_loss, perplexity, active_codes = generator(data)
    recon_loss = F.mse_loss(recon_batch, data)
    percept_loss = perceptual_loss_fn(recon_batch, data).mean()

    # dynamic adaptive weight scheduling
    if current_global_step >= disc_start_step:
        fake_outputs = discriminator(recon_batch)
        g_loss = -fake_outputs.mean()   # hinge loss

        last_layer_weights = generator.decoder.conv_out.weight
        #last_layer_weights = generator.decoder.conv3.weight
        disc_weight = calculate_adaptive_weight(recon_loss + percept_loss, g_loss, last_layer_weights)

        total_g_loss = vq_loss + recon_loss + percept_loss + (disc_weight * g_loss)
    else:
        total_g_loss = vq_loss + recon_loss + percept_loss

    total_g_loss.backward()
    optimizer_g.step()
    set_requires_grad(discriminator, True)

    # DISCRIMINATOR
    total_d_loss = torch.tensor(0.0, device=data.device)
    if current_global_step >= disc_start_step:
        optimizer_d.zero_grad()

        real_outputs = discriminator(data)
        fake_outputs_d = discriminator(recon_batch.detach())

        loss_real = F.relu(1.0 - real_outputs).mean()
        loss_fake = F.relu(1.0 + fake_outputs_d).mean()

        total_d_loss = 0.5 * loss_real + 0.5 * loss_fake
        total_d_loss.backward()
        optimizer_d.step()

    return total_g_loss.item(), total_d_loss.item(), perplexity.item(), active_codes


In [ ]:
def validate(generator, val_loader, device):
  generator.eval()
  recon_loss = 0.0
  percept_loss = 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)
    recon_batch, vq_loss = generator(data)
    recon_loss += F.mse_loss(recon_batch, data).item()
    percept_loss += perceptual_loss_fn(recon_batch, data).item()

  return recon_loss / len(val_loader), percept_loss / len(val_loader)


In [ ]:
def visual_validation(generator, val_loader, device):
  generator.eval()
  encoding_indices = []
  unique_indices = []
  visual_samples = None

  num_embeddings = generator.vq.num_embeddings
  embedding_dim = generator.vq.embedding_dim
  embeddings = generator.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      recon_batch, vq_loss = generator(data)

      visual_samples = (data.cpu(), recon_batch.cpu())

      z = generator.encoder(data)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      encoding_indices = torch.argmin(distances, dim=1)

  # percentage of used vectors
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  print(f"Total Codebook Size:    {num_embeddings}")
  print(f"Unique Vectors Used:    {len(unique_indices)} / {num_embeddings}")
  print(f"Codebook Utilization:   {util_percen:.2f}%")
  print(f"Codebook Perplexity:    {perplexity.item():.2f} (Higher is better)")

  # visual reconstruction plotting
  real_imgs, recon_imgs = visual_samples
  num_displayed_imgs = min(4, real_imgs.shape[0])

  fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

  for i in range(num_displayed_imgs):
    real_img = real_imgs[i].permute(1, 2, 0).numpy()
    recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

    real_plot = ((real_img + 1) / 2).clip(0, 1)
    recon_plot = ((recon_img + 1) / 2).clip(0, 1)

    axes[0, i].imshow(real_plot)
    axes[0, i].set_title(f"original {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_plot)
    axes[1, i].set_title(f"reconstructed {i+1}")
    axes[1, i].axis('off')

  plt.show()

In [ ]:
global_step = 0
disc_start_step = 1000
epochs = 20

save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpointstaming_transformers'

best_val_score = float('inf')

for epoch in range(epochs):
    generator.train()
    discriminator.train()
    epoch_start = time.time()

    total_loss_generator = 0.0
    perplexity_generator, avg_active_codes_generator = 0.0, 0.0
    total_loss_discriminator = 0.0

    # training
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        g_loss_val, d_loss_val, epoch_g_perplexity, epoch_g_active_codes = train_step(
            generator, discriminator, perceptual_loss_fn, data,
            optimizer_generator, optimizer_discriminator,
            disc_start_step, global_step)

        global_step += 1
        total_loss_generator += g_loss_val
        total_loss_discriminator += d_loss_val
        perplexity_generator += epoch_g_perplexity
        avg_active_codes_generator += epoch_g_active_codes

    avg_g_loss = total_loss_generator / len(train_loader)
    avg_d_loss = total_loss_discriminator / len(train_loader)
    print(f"====> Epoch {epoch} Finished | Avg G-Loss: {avg_g_loss:.4f} | Avg D-Loss: {avg_d_loss:.4f}")
    print(f"      Generator ========> Average Loss:{avg_g_loss:.4f} | Perplexity: {perplexity_generator/len(train_loader):.2f} | Active codes: {avg_active_codes_generator/len(train_loader):.2f}")
    print(f"      Discriminator ====> Average Loss:{avg_d_loss:.4f}\n")
    epoch_save_path = os.path.join(save_path, f"epoch_{epoch}")

    # save generator model at each epoch
    gen_path = os.path.join(epoch_save_path, "generator")
    os.makedirs(gen_path, exist_ok=True)

    torch.save(generator.state_dict(), os.path.join(gen_path, "main_model.pth"))
    torch.save(optimizer_generator.state_dict(), os.path.join(gen_path, "optimizer.pth"))

    # save discriminator model at each epoch
    disc_path = os.path.join(epoch_save_path, "discriminator")
    os.makedirs(disc_path, exist_ok=True)

    torch.save(discriminator.state_dict(), os.path.join(disc_path, "main_model.pth"))
    torch.save(optimizer_discriminator.state_dict(), os.path.join(disc_path, "optimizer.pth"))

    # validation
    #recon_loss, percept_loss = validate(generator, val_loader, device)
    #print(f" validation MSE: {recon_loss:.6f} | validation LPIPS: {percept_loss}")

    #if recon_loss is not None:
    #  optimizer_generator.step(recon_loss)
    #  optimizer_discriminator.step(recon_loss)
    #if epoch % 5 == 0:
    #  visual_validation(generator, val_loader, device)

    #generator.eval()
    #with torch.no_grad():


====> Epoch 0 Finished | Avg G-Loss: 1.7020 | Avg D-Loss: 0.0000
      Generator ========> Average Loss:1.7020 | Perplexity: 2.42 | Active codes: 4.61
      Discriminator ====> Average Loss:0.0000

====> Epoch 1 Finished | Avg G-Loss: 1.0992 | Avg D-Loss: 0.0000
      Generator ========> Average Loss:1.0992 | Perplexity: 3.68 | Active codes: 7.03
      Discriminator ====> Average Loss:0.0000

====> Epoch 2 Finished | Avg G-Loss: 0.8506 | Avg D-Loss: 0.8715
      Generator ========> Average Loss:0.8506 | Perplexity: 5.27 | Active codes: 8.91
      Discriminator ====> Average Loss:0.8715

====> Epoch 3 Finished | Avg G-Loss: 0.7840 | Avg D-Loss: 0.9989
      Generator ========> Average Loss:0.7840 | Perplexity: 6.17 | Active codes: 11.59
      Discriminator ====> Average Loss:0.9989

====> Epoch 4 Finished | Avg G-Loss: 0.7576 | Avg D-Loss: 1.0018
      Generator ========> Average Loss:0.7576 | Perplexity: 6.67 | Active codes: 12.73
      Discriminator ====> Average Loss:1.0018

====> Ep